1. Setup and Video Initialization

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Define paths to your specific project videos 
video_dir = 'data/videos/'
video_files = ['project_video.mp4', 'challenge_video.mp4', 'harder_challenge_video.mp4']
current_video = os.path.join(video_dir, video_files[0])

cap = cv2.VideoCapture(current_video)

In [11]:
video_dir = 'data/videos/'
video_name = 'project_video.mp4'
video_path = os.path.join(video_dir, video_name)

# Sanity Check 1: Does the file exist on disk?
if not os.path.exists(video_path):
    print(f"ERROR: Video file not found at {video_path}")
    print(f"Current working directory: {os.getcwd()}")
else:
    cap = cv2.VideoCapture(video_path)
    
    # Sanity Check 2: Did OpenCV actually open the file?
    if not cap.isOpened():
        print("ERROR: OpenCV could not open the video file. Check your codecs.")
    else:
        print(f"SUCCESS: Loaded {video_name} at {int(cap.get(cv2.CAP_PROP_FPS))} FPS")

SUCCESS: Loaded project_video.mp4 at 25 FPS


2. Pre-processing Functions

convert to grayscale and apply Gaussian smoothing to reduce noise before edge detection.

In [2]:
def preprocess_frame(frame):
    # Convert to grayscale 
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    
    # Apply Gaussian smoothing to reduce noise 
    kernel_size = 5
    blur_gray = cv2.GaussianBlur(gray, (kernel_size, kernel_size), 0)
    
    return blur_gray

def get_roi_mask(image):
    # Define a road region-of-interest (ROI) 
    mask = np.zeros_like(image)
    imshape = image.shape
    
    # Vertices for a trapezoidal ROI covering the highway lane 
    vertices = np.array([[(0, imshape[0]), (450, 320), (500, 320), (imshape[1], imshape[0])]], dtype=np.int32)
    
    cv2.fillPoly(mask, vertices, 255)
    masked_image = cv2.bitwise_and(image, mask)
    return masked_image

3. Edge and Line Detection

This implements the Canny and Hough Transform steps to find lane candidates.

In [3]:
def detect_lanes(frame):
    # 1. Preprocess and Mask
    processed = preprocess_frame(frame)
    
    # 2. Canny Edge Detection 
    low_threshold = 50
    high_threshold = 150
    edges = cv2.Canny(processed, low_threshold, high_threshold)
    
    # 3. Mask ROI 
    masked_edges = get_roi_mask(edges)
    
    # 4. Hough Line Transform 
    rho = 1              # distance resolution in pixels
    theta = np.pi/180    # angular resolution in radians
    threshold = 20       # minimum number of votes
    min_line_len = 20    # minimum number of pixels making up a line
    max_line_gap = 300   # maximum gap in pixels between connectable line segments
    
    lines = cv2.HoughLinesP(masked_edges, rho, theta, threshold, np.array([]),
                            minLineLength=min_line_len, maxLineGap=max_line_gap)
    
    return lines

4. Main Processing Loop

run the pipeline

In [17]:
# 1. Setup Video Writer 
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
out = cv2.VideoWriter('output_lane_detection.mp4', fourcc, fps, (frame_width, frame_height))
print("Processing video... Please wait.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    # --- The Pipeline Logic ---
    # 1. Detect raw lines using your detect_lanes function
    lines = detect_lanes(frame)
    
    # 2. Create an empty image for the overlay
    line_image = np.zeros_like(frame)
    
    # 3. Draw the lines (Add your averaging/extrapolation logic here later)
    if lines is not None:
        for line in lines:
            for x1, y1, x2, y2 in line:
                cv2.line(line_image, (x1, y1), (x2, y2), (0, 255, 0), 10)
    
    # 4. Merge overlay with original frame
    final_frame = cv2.addWeighted(frame, 0.8, line_image, 1, 0)
    
    # 5. Write the frame to the new file
    out.write(final_frame)

# Clean up
cap.release()
out.release()
print("Success! Video saved as 'output_lane_detection.mp4'")

Processing video... Please wait.
Success! Video saved as 'output_lane_detection.mp4'
